In [1]:
# Cell 1: Imports and Setup

import sys
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from prettytable import PrettyTable, MARKDOWN

# Add project root to path
project_root = Path('.').absolute().parent.parent
sys.path.insert(0, str(project_root))

print("Imports successful!")
print(f"Project root: {project_root}")

Imports successful!
Project root: c:\Users\nairs\Documents\GithubProjects\oWAR


C:\Users\nairs\AppData\Local\Temp\ipykernel_62264\1686519745.py:10: DeprecationWarning: the 'MARKDOWN' constant is deprecated, use the 'TableStyle' enum instead
  from prettytable import PrettyTable, MARKDOWN


In [2]:
# Cell 2: Load Projection Results

predictions_dir = project_root / 'predictions'

# Load hitter projections
hitter_path = predictions_dir / 'hitter_complete_projections_2025.csv'
if not hitter_path.exists():
    raise FileNotFoundError(
        f"Hitter projections not found: {hitter_path}\n"
        "Please run sWARm_overview.ipynb first to generate projections."
    )

hitter_complete = pd.read_csv(hitter_path)
print(f"Loaded {len(hitter_complete)} hitter projections")
print(f"Columns: {list(hitter_complete.columns[:10])}...")

# Load pitcher projections
pitcher_path = predictions_dir / 'pitcher_complete_projections_2025.csv'
if not pitcher_path.exists():
    raise FileNotFoundError(
        f"Pitcher projections not found: {pitcher_path}\n"
        "Please run sWARm_overview.ipynb first to generate projections."
    )

pitcher_complete = pd.read_csv(pitcher_path)
print(f"\nLoaded {len(pitcher_complete)} pitcher projections")
print(f"Columns: {list(pitcher_complete.columns[:10])}...")

Loaded 488 hitter projections
Columns: ['playerid', 'Name', 'Team', 'Primary_Position', 'Current_PA', 'Current_WAR', 'Current_WAR_pace', 'Remaining_PA', 'ROS_WAR', 'ROS_WAR_pace']...

Loaded 597 pitcher projections
Columns: ['playerid', 'Name', 'Team', 'Primary_Position', 'Current_IP', 'Current_WAR', 'Current_WAR_pace', 'Remaining_IP', 'ROS_WAR', 'ROS_WAR_pace']...


In [3]:
# Cell 2.5: Load Full Season 2025 Actual WAR Data

fangraphs_dir = project_root / 'MLB Player Data' / 'FanGraphs_Data'

# Load hitter full season data
hitter_full_path = fangraphs_dir / 'hitters' / 'fangraphs_hitters_2025.csv'
if hitter_full_path.exists():
    hitter_full = pd.read_csv(hitter_full_path)
    # Extract WAR and MLBAMID, rename WAR to Actual_WAR
    hitter_actual_war = hitter_full[['MLBAMID', 'WAR']].rename(columns={'WAR': 'Actual_WAR'})
    # Merge into hitter_complete (assumes playerid column matches MLBAMID)
    hitter_complete = hitter_complete.merge(
        hitter_actual_war, 
        left_on='playerid', 
        right_on='MLBAMID', 
        how='left'
    ).drop(columns=['MLBAMID'])
    print(f"Merged actual WAR for {hitter_complete['Actual_WAR'].notna().sum()} / {len(hitter_complete)} hitters")
else:
    hitter_complete['Actual_WAR'] = np.nan
    print(f"Full season hitter file not found: {hitter_full_path}")
    print("Actual WAR will not be available until full season data exists.")

# Load pitcher full season data
pitcher_full_path = fangraphs_dir / 'pitchers' / 'fangraphs_pitchers_2025.csv'
if pitcher_full_path.exists():
    pitcher_full = pd.read_csv(pitcher_full_path)
    pitcher_actual_war = pitcher_full[['MLBAMID', 'WAR']].rename(columns={'WAR': 'Actual_WAR'})
    pitcher_complete = pitcher_complete.merge(
        pitcher_actual_war,
        left_on='playerid',
        right_on='MLBAMID',
        how='left'
    ).drop(columns=['MLBAMID'])
    print(f"Merged actual WAR for {pitcher_complete['Actual_WAR'].notna().sum()} / {len(pitcher_complete)} pitchers")
else:
    pitcher_complete['Actual_WAR'] = np.nan
    print(f"Full season pitcher file not found: {pitcher_full_path}")
    print("Actual WAR will not be available until full season data exists.")

Merged actual WAR for 488 / 488 hitters
Merged actual WAR for 595 / 597 pitchers


In [4]:
# Cell 3: Filter for Players with Actual Full Season WAR Data

# Check for Actual_WAR column (full season actual WAR)
if 'Actual_WAR' in hitter_complete.columns:
    hitter_actual = hitter_complete[hitter_complete['Actual_WAR'].notna()].copy()
    print(f"Hitters with full season actual WAR: {len(hitter_actual)} / {len(hitter_complete)}")
    has_hitter_actual = True
else:
    print("Warning: No 'Actual_WAR' column found in hitter data")
    print("Available columns:", list(hitter_complete.columns))
    has_hitter_actual = False

if 'Actual_WAR' in pitcher_complete.columns:
    pitcher_actual = pitcher_complete[pitcher_complete['Actual_WAR'].notna()].copy()
    print(f"Pitchers with full season actual WAR: {len(pitcher_actual)} / {len(pitcher_complete)}")
    has_pitcher_actual = True
else:
    print("Warning: No 'Actual_WAR' column found in pitcher data")
    print("Available columns:", list(pitcher_complete.columns))
    has_pitcher_actual = False

if not has_hitter_actual and not has_pitcher_actual:
    print("\nNote: Full season data not yet available. This notebook requires 'Actual_WAR' column.")


Hitters with full season actual WAR: 488 / 488
Pitchers with full season actual WAR: 595 / 597


In [5]:
# Cell 3.5: Helper Functions for Tier Classification and Table Formatting

def classify_tier(war):
    """
    Classify player into performance tier based on full-season WAR.
    
    Tiers:
    - Elite: > 5.0 WAR
    - Good: 3.0 - 5.0 WAR
    - Average: 1.0 - 3.0 WAR
    - Below Average: 0.0 - 1.0 WAR
    - Bad: < 0.0 WAR
    """
    if war > 5.0:
        return 'elite'
    elif war >= 3.0:
        return 'good'
    elif war >= 1.0:
        return 'average'
    elif war >= 0.0:
        return 'below_average'
    else:
        return 'bad'


def create_metrics_table(metrics_dict):
    """
    Create PrettyTable for metrics display.
    
    Args:
        metrics_dict: {metric_name: value}
    
    Returns:
        PrettyTable with MARKDOWN style
    """
    table = PrettyTable()
    table.field_names = ['Metric', 'Value']
    table.set_style(MARKDOWN)
    table.align['Metric'] = 'l'
    table.align['Value'] = 'r'
    
    for metric, value in metrics_dict.items():
        table.add_row([metric, value])
    
    return table


def create_tier_metrics_table(df_with_tiers):
    """
    Create tier-based metrics table.
    
    Args:
        df_with_tiers: DataFrame with 'Tier', 'Error', 'Abs_Error' columns
    
    Returns:
        PrettyTable with metrics by tier
    """
    table = PrettyTable()
    table.field_names = ['Tier', 'Count', 'MAE', 'RMSE', 'Mean Error', 'Median Error']
    table.set_style(MARKDOWN)
    table.align['Tier'] = 'l'
    table.align['Count'] = 'r'
    table.align['MAE'] = 'r'
    table.align['RMSE'] = 'r'
    table.align['Mean Error'] = 'r'
    table.align['Median Error'] = 'r'
    
    # Calculate metrics for each tier in order
    tier_order = ['bad', 'below_average', 'average', 'good', 'elite']
    
    for tier in tier_order:
        tier_df = df_with_tiers[df_with_tiers['Tier'] == tier]
        
        if len(tier_df) == 0:
            continue
        
        count = len(tier_df)
        mae = tier_df['Abs_Error'].mean()
        rmse = np.sqrt((tier_df['Error'] ** 2).mean())
        mean_error = tier_df['Error'].mean()
        median_error = tier_df['Error'].median()
        
        table.add_row([
            tier,
            f"{count}",
            f"{mae:.3f}",
            f"{rmse:.3f}",
            f"{mean_error:+.3f}",
            f"{median_error:+.3f}"
        ])
    
    return table


def create_player_table(df, columns, title):
    """
    Create PrettyTable for top/bottom player lists.
    
    Args:
        df: DataFrame with player data
        columns: List of column names to display
        title: Table title
    
    Returns:
        PrettyTable with MARKDOWN style
    """
    table = PrettyTable()
    table.field_names = columns
    table.set_style(MARKDOWN)
    table.align = 'l'
    
    # Right-align numeric columns
    for col in ['Actual WAR', 'Projected WAR', 'Error']:
        if col in columns:
            table.align[col] = 'r'
    
    for _, row in df.iterrows():
        table.add_row([row[col] for col in columns])
    
    return table

print("Helper functions defined!")

Helper functions defined!


In [6]:
# Cell 4: Hitter Prediction Error Analysis

if has_hitter_actual:
    print("="*90)
    print("HITTER PREDICTION ERROR ANALYSIS (2025 Full Season)")
    print("="*90)
    print()
    
    # Calculate prediction error (Predicted - Actual)
    # Negative error = underestimation (model too low), Positive error = overestimation (model too high)
    hitter_actual['Error'] = hitter_actual['Total_Projected_WAR'] - hitter_actual['Actual_WAR']
    hitter_actual['Abs_Error'] = hitter_actual['Error'].abs()
    
    # Classify into tiers based on Actual_WAR
    hitter_actual['Tier'] = hitter_actual['Actual_WAR'].apply(classify_tier)
    
    # Overall metrics
    mae = hitter_actual['Abs_Error'].mean()
    rmse = np.sqrt((hitter_actual['Error'] ** 2).mean())
    mean_error = hitter_actual['Error'].mean()
    median_error = hitter_actual['Error'].median()
    
    print("OVERALL METRICS")
    print("-" * 90)
    overall_table = create_metrics_table({
        'Count': f"{len(hitter_actual)}",
        'MAE': f"{mae:.3f}",
        'RMSE': f"{rmse:.3f}",
        'Mean Error': f"{mean_error:+.3f}",
        'Median Error': f"{median_error:+.3f}"
    })
    print(overall_table)
    print()
    
    # Metrics by tier
    print("METRICS BY TIER")
    print("-" * 90)
    tier_table = create_tier_metrics_table(hitter_actual)
    print(tier_table)
    print()
    
    # Top 10 underestimates (model predicted too low - negative error)
    print("TOP 10 UNDERESTIMATES (Model predicted too LOW)")
    print("-" * 90)
    underest_df = hitter_actual.nsmallest(10, 'Error')[['Name', 'Team', 'Actual_WAR', 'Total_Projected_WAR', 'Error']].copy()
    underest_df.columns = ['Name', 'Team', 'Actual WAR', 'Projected WAR', 'Error']
    for col in ['Actual WAR', 'Projected WAR', 'Error']:
        underest_df[col] = underest_df[col].round(1)
    underest_table = create_player_table(underest_df, ['Name', 'Team', 'Actual WAR', 'Projected WAR', 'Error'], "Underestimates")
    print(underest_table)
    print()
    
    # Top 10 overestimates (model predicted too high - positive error)
    print("TOP 10 OVERESTIMATES (Model predicted too HIGH)")
    print("-" * 90)
    overest_df = hitter_actual.nlargest(10, 'Error')[['Name', 'Team', 'Actual_WAR', 'Total_Projected_WAR', 'Error']].copy()
    overest_df.columns = ['Name', 'Team', 'Actual WAR', 'Projected WAR', 'Error']
    for col in ['Actual WAR', 'Projected WAR', 'Error']:
        overest_df[col] = overest_df[col].round(1)
    overest_table = create_player_table(overest_df, ['Name', 'Team', 'Actual WAR', 'Projected WAR', 'Error'], "Overestimates")
    print(overest_table)
    print()
    print("="*90)
else:
    print("Skipping hitter error analysis (no actual WAR data)")


HITTER PREDICTION ERROR ANALYSIS (2025 Full Season)

OVERALL METRICS
------------------------------------------------------------------------------------------
| Metric       |  Value |
| :------------|------: |
| Count        |    488 |
| MAE          |  0.691 |
| RMSE         |  0.950 |
| Mean Error   | +0.170 |
| Median Error | +0.187 |

METRICS BY TIER
------------------------------------------------------------------------------------------
| Tier          | Count |   MAE |  RMSE | Mean Error | Median Error |
| :-------------|-----: |-----: |-----: |----------: |------------: |
| bad           |   153 | 0.433 | 0.585 |     +0.385 |       +0.341 |
| below_average |   125 | 0.434 | 0.600 |     +0.071 |       +0.043 |
| average       |   131 | 0.992 | 1.226 |     -0.019 |       -0.202 |
| good          |    62 | 1.139 | 1.425 |     +0.228 |       +0.008 |
| elite         |    17 | 0.939 | 1.095 |     +0.197 |       +0.219 |

TOP 10 UNDERESTIMATES (Model predicted too LOW)
-----------

In [7]:
# Cell 5: Pitcher Prediction Error Analysis

if has_pitcher_actual:
    print("="*90)
    print("PITCHER PREDICTION ERROR ANALYSIS (2025 Full Season)")
    print("="*90)
    print()
    
    # Calculate prediction error (Predicted - Actual)
    pitcher_actual['Error'] = pitcher_actual['Total_Projected_WAR'] - pitcher_actual['Actual_WAR']
    pitcher_actual['Abs_Error'] = pitcher_actual['Error'].abs()
    
    # Classify into tiers based on Actual_WAR
    pitcher_actual['Tier'] = pitcher_actual['Actual_WAR'].apply(classify_tier)
    
    # Overall metrics
    mae = pitcher_actual['Abs_Error'].mean()
    rmse = np.sqrt((pitcher_actual['Error'] ** 2).mean())
    mean_error = pitcher_actual['Error'].mean()
    median_error = pitcher_actual['Error'].median()
    
    print("OVERALL METRICS")
    print("-" * 90)
    overall_table = create_metrics_table({
        'Count': f"{len(pitcher_actual)}",
        'MAE': f"{mae:.3f}",
        'RMSE': f"{rmse:.3f}",
        'Mean Error': f"{mean_error:+.3f}",
        'Median Error': f"{median_error:+.3f}"
    })
    print(overall_table)
    print()
    
    # Metrics by tier
    print("METRICS BY TIER")
    print("-" * 90)
    tier_table = create_tier_metrics_table(pitcher_actual)
    print(tier_table)
    print()
    
    # Top 10 underestimates (model predicted too low - negative error)
    print("TOP 10 UNDERESTIMATES (Model predicted too LOW)")
    print("-" * 90)
    underest_df = pitcher_actual.nsmallest(10, 'Error')[['Name', 'Team', 'Actual_WAR', 'Total_Projected_WAR', 'Error']].copy()
    underest_df.columns = ['Name', 'Team', 'Actual WAR', 'Projected WAR', 'Error']
    for col in ['Actual WAR', 'Projected WAR', 'Error']:
        underest_df[col] = underest_df[col].round(1)
    underest_table = create_player_table(underest_df, ['Name', 'Team', 'Actual WAR', 'Projected WAR', 'Error'], "Underestimates")
    print(underest_table)
    print()
    
    # Top 10 overestimates (model predicted too high - positive error)
    print("TOP 10 OVERESTIMATES (Model predicted too HIGH)")
    print("-" * 90)
    overest_df = pitcher_actual.nlargest(10, 'Error')[['Name', 'Team', 'Actual_WAR', 'Total_Projected_WAR', 'Error']].copy()
    overest_df.columns = ['Name', 'Team', 'Actual WAR', 'Projected WAR', 'Error']
    for col in ['Actual WAR', 'Projected WAR', 'Error']:
        overest_df[col] = overest_df[col].round(1)
    overest_table = create_player_table(overest_df, ['Name', 'Team', 'Actual WAR', 'Projected WAR', 'Error'], "Overestimates")
    print(overest_table)
    print()
    print("="*90)
else:
    print("Skipping pitcher error analysis (no actual WAR data)")


PITCHER PREDICTION ERROR ANALYSIS (2025 Full Season)

OVERALL METRICS
------------------------------------------------------------------------------------------
| Metric       |  Value |
| :------------|------: |
| Count        |    595 |
| MAE          |  0.398 |
| RMSE         |  0.527 |
| Mean Error   | +0.114 |
| Median Error | +0.114 |

METRICS BY TIER
------------------------------------------------------------------------------------------
| Tier          | Count |   MAE |  RMSE | Mean Error | Median Error |
| :-------------|-----: |-----: |-----: |----------: |------------: |
| bad           |   186 | 0.354 | 0.473 |     +0.342 |       +0.234 |
| below_average |   236 | 0.318 | 0.431 |     +0.116 |       +0.054 |
| average       |   143 | 0.517 | 0.637 |     -0.207 |       -0.278 |
| good          |    24 | 0.743 | 0.845 |     +0.365 |       +0.567 |
| elite         |     6 | 0.749 | 0.835 |     -0.384 |       -0.435 |

TOP 10 UNDERESTIMATES (Model predicted too LOW)
----------

In [8]:
# Cell 6: Hitter Error Distribution Visualization

if has_hitter_actual:
    # Create error distribution histogram
    fig = go.Figure()
    
    fig.add_trace(go.Histogram(
        x=hitter_actual['Error'],
        nbinsx=50,
        name='Prediction Error',
        marker=dict(color='steelblue', line=dict(color='black', width=1))
    ))
    
    # Add vertical lines for mean and median
    mean_err = hitter_actual['Error'].mean()
    median_err = hitter_actual['Error'].median()
    
    fig.add_vline(x=0, line_dash="dash", line_color="black", 
                  annotation_text="Perfect", annotation_position="top")
    fig.add_vline(x=mean_err, line_dash="dot", line_color="red",
                  annotation_text=f"Mean: {mean_err:+.2f}", annotation_position="top right")
    fig.add_vline(x=median_err, line_dash="dot", line_color="green",
                  annotation_text=f"Median: {median_err:+.2f}", annotation_position="bottom right")
    
    fig.update_layout(
        title="Hitter Prediction Error Distribution (2025 Full Season)",
        xaxis_title="Prediction Error (Predicted - Actual WAR)",
        yaxis_title="Count",
        showlegend=False,
        height=500
    )
    
    fig.show()
else:
    print("Skipping hitter error plot (no actual data)")

In [9]:
# Cell 7: Pitcher Error Distribution Visualization

if has_pitcher_actual:
    # Create error distribution histogram
    fig = go.Figure()
    
    fig.add_trace(go.Histogram(
        x=pitcher_actual['Error'],
        nbinsx=50,
        name='Prediction Error',
        marker=dict(color='coral', line=dict(color='black', width=1))
    ))
    
    # Add vertical lines
    mean_err = pitcher_actual['Error'].mean()
    median_err = pitcher_actual['Error'].median()
    
    fig.add_vline(x=0, line_dash="dash", line_color="black",
                  annotation_text="Perfect", annotation_position="top")
    fig.add_vline(x=mean_err, line_dash="dot", line_color="red",
                  annotation_text=f"Mean: {mean_err:+.2f}", annotation_position="top right")
    fig.add_vline(x=median_err, line_dash="dot", line_color="green",
                  annotation_text=f"Median: {median_err:+.2f}", annotation_position="bottom right")
    
    fig.update_layout(
        title="Pitcher Prediction Error Distribution (2025 Full Season)",
        xaxis_title="Prediction Error (Predicted - Actual WAR)",
        yaxis_title="Count",
        showlegend=False,
        height=500
    )
    
    fig.show()
else:
    print("Skipping pitcher error plot (no actual data)")

In [10]:
# Cell 8: Actual vs Predicted Scatter Plots

if has_hitter_actual and has_pitcher_actual:
    # Create subplots
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Hitters", "Pitchers")
    )
    
    # Hitter scatter
    fig.add_trace(
        go.Scatter(
            x=hitter_actual['Actual_WAR'],
            y=hitter_actual['Total_Projected_WAR'],
            mode='markers',
            marker=dict(color='steelblue', size=8, opacity=0.6),
            text=hitter_actual['Name'],
            name='Hitters'
        ),
        row=1, col=1
    )
    
    # Pitcher scatter
    fig.add_trace(
        go.Scatter(
            x=pitcher_actual['Actual_WAR'],
            y=pitcher_actual['Total_Projected_WAR'],
            mode='markers',
            marker=dict(color='coral', size=8, opacity=0.6),
            text=pitcher_actual['Name'],
            name='Pitchers'
        ),
        row=1, col=2
    )
    
    # Add perfect prediction line to both subplots
    max_war = max(
        hitter_actual['Actual_WAR'].max(),
        hitter_actual['Total_Projected_WAR'].max(),
        pitcher_actual['Actual_WAR'].max(),
        pitcher_actual['Total_Projected_WAR'].max()
    )
    min_war = min(
        hitter_actual['Actual_WAR'].min(),
        hitter_actual['Total_Projected_WAR'].min(),
        pitcher_actual['Actual_WAR'].min(),
        pitcher_actual['Total_Projected_WAR'].min()
    )
    
    for col in [1, 2]:
        fig.add_trace(
            go.Scatter(
                x=[min_war, max_war],
                y=[min_war, max_war],
                mode='lines',
                line=dict(color='black', dash='dash'),
                showlegend=False
            ),
            row=1, col=col
        )
    
    fig.update_xaxes(title_text="Actual WAR", row=1, col=1)
    fig.update_xaxes(title_text="Actual WAR", row=1, col=2)
    fig.update_yaxes(title_text="Predicted WAR", row=1, col=1)
    fig.update_yaxes(title_text="Predicted WAR", row=1, col=2)
    
    fig.update_layout(
        title="Actual vs Predicted WAR (2025 Full Season)",
        height=500,
        showlegend=True
    )
    
    fig.show()
elif has_hitter_actual:
    print("Only hitter data available")
elif has_pitcher_actual:
    print("Only pitcher data available")
else:
    print("Skipping actual vs predicted plot (no actual data)")